# 01 — Data Pull, Inspect & Validate (Sample Databases)

**Thesis:** Implied Volatility Smile Spillovers (AP-33)  
**Author:** Başar Hacımustafaoğlu — 14866196  
**Purpose:** Pull a small sample from each accessible thesis-relevant database, inspect the structure, and validate the data quality.  

---

## What this notebook does

1. Connects to WRDS
2. For each accessible database: pulls a small sample (top 500 rows)
3. Runs full validation on each pull (shape, dtypes, missingness, date coverage, duplicates)
4. Saves raw pulls to `data/raw/` as CSV
5. Saves a validation summary report to `logs/`

**This notebook does NOT clean, merge, or analyse anything.**  
**It only answers: what does this data look like, and is it usable?**

---

> ⚠️ **Run notebook 00 first** to verify your WRDS connection.

## Step 1 — Imports and connection

In [ ]:
import os
import sys
import datetime
import json
import pandas as pd
import wrds
from dotenv import load_dotenv

sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from wrds_utils import connect_wrds, validate_df

load_dotenv()
conn = connect_wrds()

TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
RAW_DIR   = "../data/raw"
LOG_DIR   = "../logs"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f"Run timestamp: {TIMESTAMP}")

## Step 2 — Define pull targets

For each database we have access to, we specify which table to pull a sample from and why.

In [ ]:
# Each entry: (library, table, description, n_rows_to_pull)
# We pull only 500 rows per table — this is a structural inspection, not a data download

PULL_TARGETS = [
    # --- OptionMetrics US sample ---
    # 'optionprice' contains raw option prices: strike, expiry, bid, ask, volume
    # This is the rawest form of the data — what we would compute IV from
    ("optionmsamp",    "optionprice",     "OptionMetrics US — raw option prices",       500),

    # 'vsurface' contains the pre-computed volatility surface
    # Standardized IV at fixed deltas and maturities — this is what we want for smile parameterization
    ("optionmsamp",    "vsurface",        "OptionMetrics US — volatility surface",       500),

    # 'security' contains the identifier mapping (secid -> ticker/name)
    # Essential for finding SPX in the database
    ("optionmsamp",    "security",        "OptionMetrics US — security identifiers",     500),

    # --- OptionMetrics Europe sample ---
    ("optionmsampeur", "optionprice",     "OptionMetrics EU — raw option prices",        500),
    ("optionmsampeur", "vsurface",        "OptionMetrics EU — volatility surface",       500),
    ("optionmsampeur", "security",        "OptionMetrics EU — security identifiers",     500),

    # --- CBOE VIX ---
    # Daily VIX levels — our key US volatility regime control variable
    ("cboe",           "cboe_vix",        "CBOE — VIX daily levels",                    500),

    # --- Federal Reserve rates ---
    # Daily interest rates — US risk-free rate for Black-Scholes
    ("frb",            "rates_daily_ny",  "FRB — Daily rates (New York)",               500),

    # --- Fama-French factors ---
    # Daily factor returns — market risk premium and other controls
    ("ff",             "factors_daily",   "Fama-French — daily factors",                500),

    # --- CRSP sample ---
    # Daily stock/index returns — US underlying price series
    ("crspsamp",       "dsf",             "CRSP — daily stock file (sample)",           500),

    # --- Penn World Tables ---
    # Long-run macro data across countries — potential EU macro controls
    ("pwt",            "pwt100",          "Penn World Tables — main table",             500),
]

print(f"Pull targets defined: {len(PULL_TARGETS)} tables")

## Step 3 — Pull and validate each table

For each target we:
1. Pull `n` rows
2. Run validation
3. Save raw CSV to `data/raw/`
4. Log the result

Errors are caught and logged — a failed pull does not stop the notebook.

In [ ]:
pull_log = []
pulled_dfs = {}  # keep in memory for inspection below

for library, table, description, n_rows in PULL_TARGETS:

    print(f"\n{'='*60}")
    print(f"Pulling: {library}.{table}")
    print(f"Description: {description}")
    print(f"{'='*60}")

    log_entry = {
        "library":     library,
        "table":       table,
        "description": description,
        "n_requested": n_rows,
        "status":      None,
        "n_rows":      None,
        "n_cols":      None,
        "error":       None,
        "saved_to":    None,
    }

    try:
        # Pull n rows — obs parameter limits rows returned
        df = conn.get_table(library=library, table=table, obs=n_rows)

        log_entry["status"] = "SUCCESS"
        log_entry["n_rows"] = len(df)
        log_entry["n_cols"] = len(df.columns)

        # Run validation
        validate_df(df, f"{library}.{table}")

        # Save raw CSV — filename includes library, table, and timestamp
        # Never overwrite: timestamp ensures each pull is versioned
        filename = f"{library}__{table}__{TIMESTAMP}.csv"
        filepath = os.path.join(RAW_DIR, filename)
        df.to_csv(filepath, index=False)
        log_entry["saved_to"] = filepath
        print(f"\n✓ Saved to: {filepath}")

        # Keep in memory for interactive inspection
        pulled_dfs[f"{library}.{table}"] = df

    except Exception as e:
        log_entry["status"] = "ERROR"
        log_entry["error"]  = str(e)
        print(f"\n✗ ERROR: {e}")
        print("  Skipping this table and continuing.")

    pull_log.append(log_entry)

print(f"\n{'='*60}")
print("All pull attempts complete.")

## Step 4 — Pull summary

In [ ]:
summary_df = pd.DataFrame(pull_log)

print("\nPULL SUMMARY")
print("=" * 60)
print(summary_df[["library", "table", "status", "n_rows", "n_cols"]].to_string(index=False))

n_success = (summary_df["status"] == "SUCCESS").sum()
n_error   = (summary_df["status"] == "ERROR").sum()
print(f"\nSuccessful pulls : {n_success}")
print(f"Failed pulls     : {n_error}")

if n_error > 0:
    print("\nFailed tables:")
    failed = summary_df[summary_df["status"] == "ERROR"]
    for _, row in failed.iterrows():
        print(f"  {row['library']}.{row['table']}: {row['error']}")

## Step 5 — Interactive inspection

Use the cells below to look at specific tables in more detail.  
The most important ones for your thesis are the volatility surface tables.

In [ ]:
# --- Inspect OptionMetrics US volatility surface ---
# This is the most important table for your thesis

key = "optionmsamp.vsurface"
if key in pulled_dfs:
    df = pulled_dfs[key]
    print(f"OptionMetrics US Volatility Surface — first 10 rows")
    print(df.head(10).to_string())
    print(f"\nColumn names: {list(df.columns)}")
else:
    print(f"'{key}' was not pulled successfully. Check pull log above.")

In [ ]:
# --- Inspect OptionMetrics Europe volatility surface ---

key = "optionmsampeur.vsurface"
if key in pulled_dfs:
    df = pulled_dfs[key]
    print(f"OptionMetrics EU Volatility Surface — first 10 rows")
    print(df.head(10).to_string())
    print(f"\nColumn names: {list(df.columns)}")
else:
    print(f"'{key}' was not pulled successfully. Check pull log above.")

In [ ]:
# --- Inspect VIX ---

key = "cboe.cboe_vix"
if key in pulled_dfs:
    df = pulled_dfs[key]
    print(f"CBOE VIX — first 10 rows")
    print(df.head(10).to_string())
else:
    print(f"'{key}' was not pulled successfully.")

In [ ]:
# --- Inspect FRB rates ---

key = "frb.rates_daily_ny"
if key in pulled_dfs:
    df = pulled_dfs[key]
    print(f"Federal Reserve Rates — first 10 rows")
    print(df.head(10).to_string())
else:
    print(f"'{key}' was not pulled successfully.")

## Step 6 — Save full log

In [ ]:
log_path = os.path.join(LOG_DIR, f"data_pull_log_{TIMESTAMP}.json")

with open(log_path, "w") as f:
    json.dump(pull_log, f, indent=2, default=str)

print(f"Pull log saved to: {log_path}")

## Step 7 — Close connection

In [ ]:
conn.close()
print("WRDS connection closed.")
print("\n✓ Notebook 01 complete.")
print("\nNext steps:")
print("  - Review the validation reports above")
print("  - Check data/raw/ for saved CSVs")
print("  - Check logs/ for the pull log")
print("  - Note any tables that failed or returned unexpected structures")
print("  - Report findings before building notebook 02 (cleaning)")